<a href="https://colab.research.google.com/github/ritvik-123/EMP-Project/blob/master/Module_1_SmallData_Rework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentence-Level Oppression-Type Classification — Small-Data Rework

This notebook replaces the single train/test split + 2-stage cascade from
`Module_1_26.ipynb` with an approach better suited to **154 sentences / 26
groups**:

1. Smaller sentence embeddings (`bge-small-en-v1.5`, 384-dim) instead of
   `bge-large` (1024-dim) — fewer dimensions relative to your sample size.
2. **GroupKFold cross-validation** (grouped by `source_id`) instead of one
   holdout split, so every sentence gets evaluated out-of-fold and you get a
   distribution of scores instead of a single lucky/unlucky number.
3. **One single-stage multi-label classifier** instead of a gate + type
   cascade — removes the "train on an already-small subset of an
   already-small dataset" problem.
4. A **nearest-centroid baseline** trained the same way, so you can see
   whether logistic regression is actually earning its extra complexity.

Run top to bottom. Every line has a comment explaining what it does and why.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- Imports -----------------------------------------------------------
import pandas as pd                                   # tabular data handling
import numpy as np                                     # array math

from sentence_transformers import SentenceTransformer  # turns sentences into embedding vectors

from sklearn.model_selection import GroupKFold         # CV splitter that keeps a group's rows together
from sklearn.decomposition import TruncatedSVD         # dimensionality reduction (SVD works with dense/sparse)
from sklearn.linear_model import LogisticRegression    # our main classifier
from sklearn.multiclass import OneVsRestClassifier     # wraps LogisticRegression to handle multi-label output
from sklearn.metrics import classification_report      # precision/recall/F1 per label
from sklearn.preprocessing import normalize             # L2-normalizes vectors (needed for cosine similarity)

import torch


In [3]:
# --- Config --------------------------------------------------------------
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"   # your labeled sentence-level file

TARGET_LABELS = [                                # the 4 label columns we're predicting
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized",
]
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # 384-dim model: smaller than bge-large, less overfitting risk
N_SVD_COMPONENTS = 50                            # compress 384-dim embeddings down further before the classifier
N_FOLDS = 5                                      # number of GroupKFold splits (you could also use LeaveOneGroupOut)
RANDOM_STATE = 42                                # fixed seed so results are reproducible
THRESHOLD = 0.5                                  # probability cutoff for turning a score into a 0/1 prediction


In [4]:
%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


## 1. Load and clean the data

In [10]:
# --- Load -----------------------------------------------------------------
df = pd.read_csv(CSV_PATH + "Generated_Sentences_1.csv")                        # read the CSV into a DataFrame

df["Sentence"] = df["Sentence"].fillna("").astype(str).str.strip()   # replace NaNs with "", strip whitespace
df = df[df["Sentence"] != ""].reset_index(drop=True)                 # drop any genuinely empty sentences

for label in TARGET_LABELS:                                          # loop over each of the 4 label columns
    df[label] = pd.to_numeric(df[label], errors="coerce")            # force numeric, turn bad values into NaN
    df[label] = df[label].fillna(0).clip(0, 1).astype(int)           # NaN -> 0, clip to {0,1}, cast to int

print("Rows after cleaning:", len(df))                               # sanity check row count
print("Unique groups (source_id):", df["source_id"].nunique())       # sanity check group count
print(df[TARGET_LABELS].sum())                                       # how many positives per label


KeyError: 'ideological'

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Embed the sentences

We embed the raw sentence text once, up front, for the whole dataset. Note
that fitting the classifier still happens *inside* each CV fold — the
embedding step itself doesn't "see" the labels, so computing it once outside
the loop is not a leak.

In [7]:
# --- Embed ------------------------------------------------------------------
device = "cuda"   # change to "cpu" if you're not on a GPU runtime

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)   # loads the pretrained embedding model

sentences = df["sentence"].tolist()                                   # plain list of strings to embed

X_full = embedder.encode(
    sentences,                     # the sentences to embed
    batch_size=32,                 # how many sentences to embed per forward pass
    normalize_embeddings=True,     # L2-normalize each embedding (helps cosine-similarity-style comparisons)
    show_progress_bar=True,        # display a progress bar since this can take a minute
    convert_to_numpy=True,         # return a numpy array instead of torch tensors
)

print("Embedding matrix shape:", X_full.shape)   # expect (num_sentences, 384)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding matrix shape: (154, 384)


## 3. Reduce dimensionality with SVD

384 dimensions on ~150 training rows is still a lot of parameters relative to
data. `TruncatedSVD` compresses the embeddings to `N_SVD_COMPONENTS` dims,
which reduces overfitting risk in the downstream logistic regression.

**Important**: SVD is fit *inside* each CV fold (see the loop below), not on
the whole dataset up front — fitting it globally first would leak
information from the test fold into training.

## 4. Prepare labels and groups for cross-validation

In [8]:
# --- Targets and groups -----------------------------------------------------
Y = df[TARGET_LABELS].values          # shape (n_sentences, 4) — the multi-label target matrix
groups = df["source_id"].values       # which participant/source each sentence belongs to

# GroupKFold ensures all sentences from the same source_id land in the same
# fold — so the model is never trained and tested on sentences from the same
# person, which would make scores look better than they really are.
gkf = GroupKFold(n_splits=N_FOLDS)


## 5. Cross-validated logistic regression (single-stage, multi-label)

Instead of a gate stage + a separate type stage, this trains **one**
`OneVsRestClassifier(LogisticRegression)` per fold on all 4 labels at once. A
sentence with all-zero predictions is implicitly "no strong leaning" — you
don't need a separate gate model for that.

In [9]:
# --- Cross-validated training loop -------------------------------------------
oof_proba_lr = np.zeros_like(Y, dtype=float)   # will hold out-of-fold predicted probabilities for every row

fold_reports = []   # collect per-fold classification reports so we can see variance across folds

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, Y, groups=groups)):
    # --- split this fold's data ---
    X_train_raw, X_test_raw = X_full[train_idx], X_full[test_idx]   # raw 384-dim embeddings for train/test rows
    Y_train, Y_test = Y[train_idx], Y[test_idx]                     # matching label rows

    # --- fit SVD on TRAIN ONLY, then apply to both train and test ---
    svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=RANDOM_STATE)  # dimensionality reducer
    X_train = svd.fit_transform(X_train_raw)   # learn the SVD components from training embeddings only
    X_test = svd.transform(X_test_raw)         # apply the same learned components to the test embeddings

    # --- fit the classifier on this fold's training data ---
    clf = OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,              # more iterations so the solver reliably converges
            class_weight="balanced",    # up-weights rare positive labels automatically
            solver="liblinear",         # solver that works well for small/medium binary problems
            random_state=RANDOM_STATE,  # reproducibility
        )
    )
    clf.fit(X_train, Y_train)                      # train on this fold's training rows

    # --- predict probabilities on the held-out fold ---
    test_proba = clf.predict_proba(X_test)          # shape (n_test_rows, 4) — probability per label
    oof_proba_lr[test_idx] = test_proba             # store these as this fold's out-of-fold predictions

    # --- per-fold report, just to see how much folds vary ---
    fold_preds = (test_proba >= THRESHOLD).astype(int)
    fold_reports.append(
        classification_report(Y_test, fold_preds, target_names=TARGET_LABELS,
                               zero_division=0, output_dict=True)
    )
    print(f"Fold {fold_idx}: {len(test_idx)} test rows, {len(train_idx)} train rows")

print("\nDone. Every row now has an out-of-fold probability it never trained on.")


Fold 0: 31 test rows, 123 train rows
Fold 1: 31 test rows, 123 train rows
Fold 2: 31 test rows, 123 train rows
Fold 3: 31 test rows, 123 train rows
Fold 4: 30 test rows, 124 train rows

Done. Every row now has an out-of-fold probability it never trained on.


## 6. Aggregate results across all folds

This is the single number/report you should actually cite — every sentence
was scored by a model that never saw it during training, and it pools all
26 groups instead of relying on one lucky/unlucky split.

In [10]:
# --- Pooled out-of-fold evaluation -------------------------------------------
oof_preds_lr = (oof_proba_lr >= THRESHOLD).astype(int)   # threshold the pooled probabilities into 0/1

print("=== Logistic Regression: pooled out-of-fold report (all 26 groups) ===")
print(classification_report(Y, oof_preds_lr, target_names=TARGET_LABELS, zero_division=0))


=== Logistic Regression: pooled out-of-fold report (all 26 groups) ===
                   precision    recall  f1-score   support

      ideological       0.34      0.55      0.42        22
institutionalized       0.12      0.23      0.15        13
    interpersonal       0.44      0.59      0.51        27
     internalized       0.22      0.43      0.29        21

        micro avg       0.29      0.48      0.36        83
        macro avg       0.28      0.45      0.34        83
     weighted avg       0.31      0.48      0.37        83
      samples avg       0.13      0.17      0.14        83



In [11]:
# --- Per-fold variance check --------------------------------------------------
# Pull out each fold's macro-F1 so you can see how much a single split would
# have swung your reported number.
macro_f1_per_fold = [r["macro avg"]["f1-score"] for r in fold_reports]   # macro-F1 for each of the 5 folds

print("Macro-F1 per fold:", np.round(macro_f1_per_fold, 3))
print("Mean macro-F1:", np.round(np.mean(macro_f1_per_fold), 3))
print("Std  macro-F1:", np.round(np.std(macro_f1_per_fold), 3))
print("--> report mean ± std, not a single split's number, in any writeup.")


Macro-F1 per fold: [0.522 0.205 0.174 0.263 0.375]
Mean macro-F1: 0.308
Std  macro-F1: 0.127
--> report mean ± std, not a single split's number, in any writeup.


## 7. Nearest-centroid baseline (sanity check)

With only 13-27 positive examples per label, it's worth checking whether
logistic regression is actually beating the simplest possible model: for
each label, compute the mean embedding of its positive examples (the
"centroid"), then classify a test sentence by its cosine similarity to that
centroid. If logistic regression doesn't clearly beat this, the extra
complexity isn't buying you anything at this sample size.

In [12]:
# --- Centroid classifier, cross-validated the same way -----------------------
oof_proba_centroid = np.zeros_like(Y, dtype=float)   # out-of-fold "probabilities" (cosine similarities) for centroid model

for train_idx, test_idx in gkf.split(X_full, Y, groups=groups):
    X_train_raw, X_test_raw = X_full[train_idx], X_full[test_idx]   # raw embeddings for this fold
    Y_train = Y[train_idx]                                          # training labels for this fold

    # L2-normalize so dot product == cosine similarity
    X_train_norm = normalize(X_train_raw)
    X_test_norm = normalize(X_test_raw)

    for label_idx, label in enumerate(TARGET_LABELS):
        positive_mask = Y_train[:, label_idx] == 1              # which training rows are positive for this label
        if positive_mask.sum() == 0:                            # guard against a fold with zero positives
            continue                                            # skip — leave scores at 0 for this label/fold
        centroid = X_train_norm[positive_mask].mean(axis=0)     # average embedding of positive examples
        centroid = centroid / np.linalg.norm(centroid)          # re-normalize the averaged vector to unit length
        sims = X_test_norm @ centroid                           # cosine similarity of each test row to the centroid
        oof_proba_centroid[test_idx, label_idx] = sims          # store as this label's "score" for these rows

# turn similarities into 0/1 predictions using the same-shaped threshold logic
# (similarities range roughly -1..1, so we pick a per-label threshold at the median positive-class similarity
#  from the pooled scores, purely as a simple, comparable cutoff)
centroid_threshold = np.median(oof_proba_centroid[Y == 1])   # rough shared threshold across labels
oof_preds_centroid = (oof_proba_centroid >= centroid_threshold).astype(int)

print("=== Nearest-Centroid baseline: pooled out-of-fold report ===")
print(classification_report(Y, oof_preds_centroid, target_names=TARGET_LABELS, zero_division=0))


=== Nearest-Centroid baseline: pooled out-of-fold report ===
                   precision    recall  f1-score   support

      ideological       0.24      0.59      0.34        22
institutionalized       0.16      0.46      0.24        13
    interpersonal       0.24      0.44      0.31        27
     internalized       0.20      0.52      0.29        21

        micro avg       0.21      0.51      0.30        83
        macro avg       0.21      0.51      0.29        83
     weighted avg       0.21      0.51      0.30        83
      samples avg       0.08      0.18      0.11        83



## 8. Compare the two models side by side

If logistic regression's macro-F1 isn't meaningfully above the centroid
baseline's, that's a signal your embeddings + labels don't yet support
anything more complex than "average embedding of known examples" — which is
a completely legitimate, honest thing to report in a small-data study.

In [13]:
from sklearn.metrics import f1_score   # macro-F1 helper for the final side-by-side comparison

lr_macro_f1 = f1_score(Y, oof_preds_lr, average="macro", zero_division=0)              # LR pooled macro-F1
centroid_macro_f1 = f1_score(Y, oof_preds_centroid, average="macro", zero_division=0)   # centroid pooled macro-F1

print("Logistic Regression macro-F1 (pooled OOF):", round(lr_macro_f1, 3))
print("Nearest-Centroid macro-F1   (pooled OOF):", round(centroid_macro_f1, 3))


Logistic Regression macro-F1 (pooled OOF): 0.343
Nearest-Centroid macro-F1   (pooled OOF): 0.292


## Notes on what changed vs. the original notebook

- **No single held-out split** — every sentence contributes to both training
  (in 4 of 5 folds) and evaluation (in its 1 held-out fold), and the report
  you cite is pooled across all folds, not one split.
- **No gate + type cascade** — one multi-label model handles "is this
  oppression-related" and "which type" simultaneously; an all-zero row is
  implicitly "no strong leaning."
- **SVD is refit inside each fold** — this avoids leaking test-fold
  information into the dimensionality reduction step.
- **A trivial baseline (nearest centroid) is reported alongside the real
  model** — at n≈150 this is the honest way to show your logistic
  regression is earning its complexity, not just fitting noise.
